[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1_EUrGaZQqAXsiTf9YGq96H4uqG18igw-/view?usp=drive_link)

# Prompt Evaluation – Custom Metrics

This notebook demonstrates how to add custom metrics and criteria-based scoring to prompt evaluations. You can use function-based metrics or LLM-as-judge criteria to compare prompt quality.

**Objectives**
- Install Floeval and configure credentials
- Define a custom metric for prompt evaluation
- Use `criteria()` for LLM-as-judge scoring
- Run evaluation and compare results across prompts

## 1. Installation

Install Floeval before running this notebook.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
%pip install git+https://github.com/FloTorch/floeval.git@dev

## 2. Configuration Constants

Set the following constants before running. Replace placeholder values with your API credentials and model identifiers. Required only for the optional criteria-based metric; function-based metrics do not require an API key.

**Provider flexibility:** You can use any OpenAI-compatible provider (OpenAI, Azure OpenAI, Anthropic, local models, etc.) — set the appropriate `base_url` and model names for your provider.

**Using FloTorch:** If you want to use FloTorch keys and gateway, obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
import getpass

# LLM and API configuration (OpenAI)
OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_API_KEY = getpass.getpass("Enter your API key: ")
OPENAI_CHAT_MODEL = "gpt-4o-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

## 1. Create the Prompts File

A YAML file is created with three prompt variants (IDs 1, 2, 3) for a support-style scenario.

In [ ]:
from pathlib import Path

prompts_content = """
prompts:
  "1":
    template: "Respond with empathy and understanding."
  "2":
    template: "Respond directly and professionally."
  "3":
    template: "Respond in a balanced tone, acknowledging concerns while staying solution-focused."
"""
Path("prompts_custom.yaml").write_text(prompts_content.strip())
print("Created prompts_custom.yaml")

## 2. Imports

Import `Evaluation`, `DatasetLoader`, `custom_metric`, `criteria`, and the LLM config.

In [ ]:
from floeval import Evaluation, DatasetLoader
from floeval.api.metrics.custom import custom_metric, criteria
from floeval.config.schemas.io.llm import OpenAIProviderConfig

## 3. Define a Custom Metric (Response Length)

A function-based metric is defined using the `@custom_metric` decorator. No API key is required.

In [ ]:
@custom_metric(name="response_length", threshold=0.3)
def response_length(response: str) -> float:
    """Score 0–1 based on response length (capped at 100 chars)."""
    return min(len(response) / 100.0, 1.0)

## 4. Define a Criteria-Based Metric (Empathy)

The `criteria()` helper is used to define an LLM-as-judge metric. This requires `llm_config` and an API key.

In [ ]:
empathy = criteria(
    name="empathy",
    description="Rate empathy from 0 to 1. Reward acknowledgment and supportive tone.",
    threshold=0.6,
)

## 5. Load the Dataset and Run Evaluation

A partial dataset with `prompt_ids` is loaded. The evaluation is run with both the custom metric and the criteria-based metric.

In [ ]:
dataset = DatasetLoader.from_samples(
    [
        {"user_input": "Customer is upset about a delayed order.", "prompt_ids": ["1", "2", "3"]},
    ],
    partial_dataset=True,
)

llm_config = OpenAIProviderConfig(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    chat_model=OPENAI_CHAT_MODEL,
    embedding_model=OPENAI_EMBEDDING_MODEL,
)

evaluation = Evaluation(
    dataset=dataset,
    llm_config=llm_config,
    metrics=["custom:response_length", empathy],
    default_provider="ragas",
    dataset_generator_model=OPENAI_CHAT_MODEL,
    prompts_file="prompts_custom.yaml",
)

results = evaluation.run()
print("Aggregate scores:", results.aggregate_scores)

## 6. Inspect Results by Prompt

Each result includes `prompt_id` and Results are inspected by prompt to compare faithfulness across instructions.

In [ ]:
for sr in results.sample_results:
    pid = sr.get("prompt_id", "unknown")
    metrics = sr.get("metrics", {})
    print(f"Prompt: {pid}")
    for k, v in metrics.items():
        print(f"  {k}: {v.get('score')}")

## Summary

This notebook demonstrated how to add custom and criteria-based metrics to prompt evaluations.

The key components included:

1. **Prompts File**: A YAML file with three prompt variants (IDs 1, 2, 3) for support-style scenarios was created.
2. **Custom Metric**: A function-based `response_length` metric was defined using `@custom_metric`.
3. **Criteria-Based Metric**: The `criteria()` helper was used to define an `empathy` LLM-as-judge metric.
4. **Evaluation Execution**: The prompt evaluation was run with both metrics and aggregate scores were inspected.

This example showcases combining custom, criteria-based, and built-in metrics in prompt evaluation.